In [ ]:
%load_ext jupyter_tikz
%load_ext watermark


In [ ]:
import hashlib
import re
import xml.etree.ElementTree as xml_etree


import dendropy as dp
from IPython.display import display, HTML
from hstrat import _auxiliary_lib as hstrat_aux
import matplotlib as mpl
from matplotlib import pyplot as plt
import polars as pl
from slugify import slugify
import svgpath2mpl
from teeplot import teeplot as tp

from pylib._seed_global_rngs import seed_global_rngs


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = "2025-06-03-vanilla-treeviz-dendropy"
teeplot_subdir


In [ ]:
seed_global_rngs(1)


## Get Data


In [ ]:
url = "https://osf.io/r8skg/download"
tmp_path = f"/tmp/{teeplot_subdir}.pqt"

print(f"Downloading data from {url}")

pl.scan_parquet(
    url,
    low_memory=True,
    retries=5,
).sink_parquet(tmp_path)
print("done!")


In [ ]:
df = pl.scan_parquet(
    tmp_path,
    low_memory=True,
    retries=5,
)
schema = df.collect_schema()


In [ ]:
fil = (
    df.filter(
        pl.col("trt_hsurf_bits").eq(0),
    )
    .filter(
        pl.col("replicate_uuid").eq(
            pl.col("replicate_uuid").first().over("trt_name"),
        )
    )
    .select(
        pl.exclude([k for k, v in schema.items() if v == pl.String]),
    )
    .collect()
)


In [ ]:
def display_tikz_plot_mpl(
    tree: dp.Tree, *args: list, **kwargs: dict
) -> mpl.figure.Figure:
    tikz_svg = tree.display_tikz_plot(*args, **kwargs)
    print(f"{hashlib.sha256(tikz_svg.data.encode('utf-8')).hexdigest()=}")
    print(f"{len(tikz_svg.data)=}")

    # adapted from https://nbviewer.org/github/nvictus/svgpath2mpl/blob/master/examples/homer.ipynb
    root = xml_etree.fromstring(
        tikz_svg.data.replace("rgb(0%, 0%, 0%)", "black"),
    )
    width = int(re.match(r"\d+", root.attrib["width"]).group())
    height = int(re.match(r"\d+", root.attrib["height"]).group())
    path_elems = root.findall(r".//{http://www.w3.org/2000/svg}path")

    paths = [
        svgpath2mpl.parse_path(elem.attrib["d"]) for elem in path_elems
    ]
    print(f"{len(paths)=}")

    edgecolors = [elem.attrib.get("stroke", "none") for elem in path_elems]
    facecolors = [elem.attrib.get("fill", "none") for elem in path_elems]
    linewidths = [
        elem.attrib.get("stroke_width", 1) for elem in path_elems
    ]
    collection = mpl.collections.PathCollection(
        paths,
        edgecolors=edgecolors,
        facecolors=facecolors,
        linewidths=linewidths,
    )

    fig, ax = plt.subplots()
    ax.set_xlim([0, width])
    ax.set_ylim([height, 0])
    collection.set_transform(ax.transData)
    ax.add_artist(collection)
    return fig


## Plot Data


In [ ]:
for (trt_name,), group in fil.group_by("trt_name"):
    display(HTML(f"<h1>{trt_name}</h1>"))

    phylo_df = group.to_pandas()

    phylo_df = hstrat_aux.alifestd_mark_num_descendants_asexual(phylo_df)
    phylo_df = hstrat_aux.alifestd_mark_clade_duration_asexual(phylo_df)
    phylo_df[
        "calc_clade_trait_count_asexual"
    ] = hstrat_aux.alifestd_calc_clade_trait_count_asexual(
        phylo_df, trait_mask=phylo_df["is_focal_defmut"].values
    )

    mask = (
        (phylo_df["origin_time"] > 50)
        & (phylo_df["num_descendants"] >= 500)
        & (phylo_df["num_descendants"] <= 750)
        & (phylo_df["calc_clade_trait_count_asexual"] > 0)
    )
    sample_idx = phylo_df.loc[mask, "id"].sample(n=1, random_state=1).item()
    sample_mask = phylo_df["id"] == sample_idx

    phylo_df = hstrat_aux.alifestd_mask_descendants_asexual(
        phylo_df,
        ancestor_mask=phylo_df["is_focal_defmut"],
    ).rename(
        columns={
            "alifestd_mask_descendants_asexual": "focal_defmut_descendant",
        },
    )

    phylo_df = hstrat_aux.alifestd_mask_descendants_asexual(
        phylo_df,
        ancestor_mask=sample_mask,
    )

    phylo_df["extant"] = phylo_df["alifestd_mask_descendants_asexual"]
    print(f"{phylo_df['extant'].sum()=}")
    phylo_df = hstrat_aux.alifestd_prune_extinct_lineages_asexual(phylo_df)

    sample_ot_min = phylo_df.loc[
        phylo_df["extant"].values, "origin_time"
    ].min()
    sample_ot_max = phylo_df.loc[
        phylo_df["extant"].values, "origin_time"
    ].max()

    newick = hstrat_aux.alifestd_as_newick_asexual(
        phylo_df,
        taxon_label="focal_defmut_descendant",
    )

    tree = dp.Tree.get(
        data=newick,
        schema="newick",
        suppress_internal_node_taxa=True,
        suppress_leaf_node_taxa=True,
    )

    # way 1, via matplotlib
    tp.tee(
        display_tikz_plot_mpl,
        tree,
        node_label_compose_fn=lambda n: "",
        teeplot_outattrs={"trt_name": slugify(trt_name)},
    )

    # way 2, pure svg
    tikz_svg = tree.display_tikz_plot(node_label_compose_fn=lambda n: "")
    tikz_svg.data = tikz_svg.data.replace(
        "<svg",
        "<svg style='background-color: white'",
    )
    display(tikz_svg)
